# l12_links_1 seed 42, cohort_safe inner validation

Same arm, same outer folds (`SPLIT_SEED=42`, 1,066 sections, 7,461,039 cells). The new run changes only the early-stopping split to `INNER_VAL=cohort_safe`. The row beside it is the earlier random-split `l12_links_1` seed 42. Both selected weights are fold 4. This notebook reads the pooled out-of-fold files. It does not call predict.

The number to compare is the pooled OOF L2 macro AUROC in `validation_internal_metrics.csv` (the log line `OOF L2 macro AUROC=`). Decision columns are cross-fitted `decision full` from `decision_report.csv`. Fold bars are the five held-out folds, not that one checkpoint.

Helpers live in `pred_statistic_visual.py`. The older A0–A3 comparison stays in `2_Pred_statistic_visual_pancancer_uni2_ablation.ipynb`.


In [ ]:
import sys
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if os.environ.get("HIST2PHENO_REPO"):
    REPO = Path(os.environ["HIST2PHENO_REPO"]).expanduser().resolve()
else:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "code" / "PanCancer").is_dir():
            REPO = candidate
            break
PKG = REPO / "code" / "Hist2Pheno_pkg"
if str(PKG) not in sys.path:
    sys.path.insert(0, str(PKG))
PAN = REPO / "code" / "PanCancer"
if str(PAN) not in sys.path:
    sys.path.insert(0, str(PAN))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

from pred_statistic_visual import fold_csv, load_folds

PAN_DATA = REPO / "data" / "PanCancer"
OUT = PAN_DATA / "l12_links_1_ivsafe_seed42_compare"
OUT.mkdir(parents=True, exist_ok=True)

RUNS = {
    "cohort_safe": {
        "label": "cohort_safe seed 42",
        "color": "seagreen",
        "result": PAN_DATA / "result_uni2_ablation_l12_links_1_new1066axisfix_ivsafe_paired_v2_seed42",
        "tag": "l12_links_1",
    },
    "random": {
        "label": "random seed 42",
        "color": "steelblue",
        "result": PAN_DATA / "result_uni2_ablation_l12_links_1_new1066axisfix_paired_v2_seed42",
        "tag": "l12_links_1",
    },
}


def decision_l2(result: Path) -> dict:
    report = pd.read_csv(result / "decision_report.csv")
    block = report.loc[(report["cohort"] == "ALL") & (report["head"] == "L2")]
    out = {}
    for method, key in (
        ("argmax per head (current)", "argmax"),
        ("decision full (cross-fitted)", "decision"),
    ):
        row = block.loc[block["method"] == method].iloc[0]
        out[f"L2_acc_{key}"] = float(row["accuracy"])
        out[f"L2_f1_{key}"] = float(row["macro_f1"])
    return out


ready = []
inv_rows = []
for key, spec in RUNS.items():
    ckpt = spec["result"] / "cross_dataset_cv" / spec["tag"]
    metrics_path = spec["result"] / "validation_internal_metrics.csv"
    fold_path = fold_csv(spec)
    has_oof = metrics_path.is_file() and fold_path.is_file()
    inv_rows.append({
        "run": key,
        "label": spec["label"],
        "best_mlp": (ckpt / "best_mlp_gpu.pt").is_file(),
        "n_fold_ckpt": len(list(ckpt.glob("hce_group_fold_[0-9].pt"))),
        "oof_npz": (spec["result"] / "oof_predictions.npz").is_file(),
        "pooled_oof": has_oof,
        "decision": (spec["result"] / "decision_report.csv").is_file(),
    })
    if has_oof:
        ready.append(key)
    else:
        print(f"{key}: pooled OOF not written yet ({metrics_path})")
display(pd.DataFrame(inv_rows))
if "cohort_safe" not in ready:
    raise FileNotFoundError("cohort_safe seed 42 has no pooled OOF")

rows = []
for key in ready:
    spec = RUNS[key]
    row = pd.read_csv(spec["result"] / "validation_internal_metrics.csv").iloc[0].to_dict()
    folds_raw = pd.read_csv(fold_csv(spec))
    row.update({
        "run": key,
        "label": spec["label"],
        "mean_fold_l2_auc": float(folds_raw["l2_macro_auc"].mean()),
        "mean_fold_l12_auc": float(folds_raw["l12_macro_auc"].mean()),
        "mean_fold_l1_auc": float(folds_raw["l1_head_macro_auc"].mean()),
    })
    if (spec["result"] / "decision_report.csv").is_file():
        row.update(decision_l2(spec["result"]))
    rows.append(row)
summary = pd.DataFrame(rows)
cols = [c for c in (
    "label", "n_cells", "best_fold", "oof_l2_macro_auc", "cv_l2_macro_f1_mean",
    "mean_fold_l2_auc", "mean_fold_l12_auc", "mean_fold_l1_auc",
    "L2_acc_argmax", "L2_f1_argmax", "L2_acc_decision", "L2_f1_decision",
) if c in summary.columns]
display(summary[cols])
summary[cols].to_csv(OUT / "pooled_oof.csv", index=False)

if set(ready) >= {"cohort_safe", "random"}:
    new = summary.loc[summary["run"] == "cohort_safe"].iloc[0]
    old = summary.loc[summary["run"] == "random"].iloc[0]
    print(
        f"cohort_safe minus random   "
        f"pooled L2 AUROC {float(new.oof_l2_macro_auc) - float(old.oof_l2_macro_auc):+.4f}   "
        f"argmax L2 F1 {float(new.L2_f1_argmax) - float(old.L2_f1_argmax):+.4f}   "
        f"decision L2 F1 {float(new.L2_f1_decision) - float(old.L2_f1_decision):+.4f}"
    )

for key in ready:
    spec = RUNS[key]
    folds_raw = pd.read_csv(fold_csv(spec))
    if "inner_lost_classes" not in folds_raw.columns:
        print(f"{spec['label']}: group_cv_folds.csv has no inner-val audit")
        continue
    lost = folds_raw["inner_lost_classes"].fillna("").astype(str)
    sections = folds_raw.get("inner_val_sections", pd.Series([""] * len(folds_raw))).fillna("").astype(str)
    print(
        f"{spec['label']}: inner training lost an L2 class in {int((lost != '').sum())}/{len(folds_raw)} folds; "
        f"BRCA section in inner val {int(sections.str.contains('BRCA').sum())}/{len(folds_raw)}"
    )

folds = load_folds(fold_csv(RUNS[ready[0]]), ready[0])
for key in ready[1:]:
    folds = folds.join(load_folds(fold_csv(RUNS[key]), key))
folds = folds.reset_index()
display(folds)
folds.to_csv(OUT / "fold_oof.csv", index=False)

n_run = len(ready)
width = 0.8 / n_run
auc_cols = [f"{key}_L2_auc" for key in ready] + [f"{key}_L12_auc" for key in ready] + [f"{key}_L1_auc" for key in ready]
auc_low = float(folds[auc_cols].min().min()) - 0.01
auc_high = float(folds[auc_cols].max().max()) + 0.01
fig, axes = plt.subplots(1, 3, figsize=(9.2, 3.2), sharey=True)
for ax, metric, title in (
    (axes[0], "L1_auc", "L1 head"),
    (axes[1], "L12_auc", "L12"),
    (axes[2], "L2_auc", "L2"),
):
    x = folds["fold"].to_numpy()
    for i, key in enumerate(ready):
        ax.bar(
            x + (i - (n_run - 1) / 2) * width,
            folds[f"{key}_{metric}"],
            width=width,
            label=RUNS[key]["label"],
            color=RUNS[key]["color"],
        )
    ax.set_title(f"{title} macro-AUROC")
    ax.set_xlabel("fold")
    ax.set_xticks(list(range(5)))
    ax.set_ylim(auc_low, auc_high)
    ax.legend(fontsize=7)
axes[0].set_ylabel("AUROC")
fig.suptitle("Fold OOF macro-AUROC: l12_links_1 seed 42", y=1.02)
fig.tight_layout()
fig.savefig(OUT / "fold_oof_auroc.pdf", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4.2, 3.4))
x = folds["fold"].to_numpy()
for i, key in enumerate(ready):
    ax.bar(
        x + (i - (n_run - 1) / 2) * width,
        folds[f"{key}_L2_f1"],
        width=width,
        label=RUNS[key]["label"],
        color=RUNS[key]["color"],
    )
ax.set_xticks(list(range(5)))
ax.set_xlabel("fold")
ax.set_ylabel("L2 macro-F1")
ax.set_title("Fold OOF L2 macro-F1")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUT / "fold_oof_l2_f1.pdf", dpi=150, bbox_inches="tight")
plt.show()

for key in ready:
    spec = RUNS[key]
    display(Markdown(f"## {spec['label']}"))
    for title, stem in (("OOF L2 ROC", "roc_internal_level2_oof"), ("OOF L2 confusion", "conf_matrix_level2")):
        path = next((spec["result"] / f"{stem}{suffix}" for suffix in (".png", ".pdf") if (spec["result"] / f"{stem}{suffix}").is_file()), None)
        if path is None:
            print(f"{key} missing {stem}")
            continue
        print(title, path.name)
        display(Image(filename=str(path), width=480))


In [ ]:
# Per-class F1 and per-cohort L2, from the decision files already written with the OOF.
parts = []
for key in ready:
    table = pd.read_csv(RUNS[key]["result"] / "decision_per_class_f1.csv")
    table = table.rename(columns={
        "f1_argmax": f"f1_argmax_{key}",
        "f1_decision_full": f"f1_decision_{key}",
    })
    keep = ["class", "support", f"f1_argmax_{key}", f"f1_decision_{key}"]
    parts.append(table[keep])
per_class = parts[0]
for part in parts[1:]:
    per_class = per_class.merge(part.drop(columns=["support"]), on="class")
if "f1_argmax_random" in per_class.columns:
    per_class["argmax_minus_random"] = per_class["f1_argmax_cohort_safe"] - per_class["f1_argmax_random"]
    per_class["decision_minus_random"] = per_class["f1_decision_cohort_safe"] - per_class["f1_decision_random"]
per_class = per_class.sort_values("support", ascending=False)
display(per_class)
per_class.to_csv(OUT / "per_class_f1.csv", index=False)

fig, ax = plt.subplots(figsize=(7.2, 0.32 * len(per_class) + 1.2))
y = range(len(per_class))
height = 0.8 / len(ready)
for i, key in enumerate(ready):
    ax.barh(
        [v + (i - (len(ready) - 1) / 2) * height for v in y],
        per_class[f"f1_argmax_{key}"],
        height=height,
        label=RUNS[key]["label"],
        color=RUNS[key]["color"],
    )
ax.set_yticks(list(y), per_class["class"])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("OOF L2 F1, argmax")
ax.set_title("Per-class OOF L2 F1")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "per_class_l2_f1.pdf", dpi=150, bbox_inches="tight")
plt.show()

cohort_parts = []
for key in ready:
    report = pd.read_csv(RUNS[key]["result"] / "decision_report.csv")
    block = report.loc[
        (report["head"] == "L2")
        & (report["cohort"] != "ALL")
        & (report["method"] == "argmax per head (current)")
    ].copy()
    block = block.rename(columns={"accuracy": f"acc_{key}", "macro_f1": f"f1_{key}"})
    cohort_parts.append(block[["cohort", "n_cells", f"acc_{key}", f"f1_{key}"]])
by_cohort = cohort_parts[0]
for part in cohort_parts[1:]:
    by_cohort = by_cohort.merge(part.drop(columns=["n_cells"]), on="cohort")
by_cohort = by_cohort.sort_values("n_cells", ascending=False)
display(by_cohort)
by_cohort.to_csv(OUT / "per_cohort_l2_argmax.csv", index=False)

fig, ax = plt.subplots(figsize=(7.2, 3.4))
x = range(len(by_cohort))
for i, key in enumerate(ready):
    ax.bar(
        [v + (i - (len(ready) - 1) / 2) * width for v in x],
        by_cohort[f"f1_{key}"],
        width=width,
        label=RUNS[key]["label"],
        color=RUNS[key]["color"],
    )
ax.set_xticks(list(x), by_cohort["cohort"], rotation=0)
ax.set_ylabel("L2 macro-F1, argmax")
ax.set_title("Per-cohort OOF L2 macro-F1")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(OUT / "per_cohort_l2_f1.pdf", dpi=150, bbox_inches="tight")
plt.show()
